# Preparation database "res. 0.25°"

In [ ]:
# Libraries
import os, shutil
import numpy as np
import xarray as xr
import dask.dataframe as dd
import matplotlib.pyplot as plt

In [ ]:
# Directories
dir01 = '../paper_ecosystem_carbon/output/01_data_prep/'
dir02 = '../paper_ecosystem_carbon/output/02_database_prep/'

---

In [ ]:
# Land mask
ds_land = xr.open_dataset(f'{dir01}ds_luh2_land_mask_r025.nc')

# Soilgrids 2017
ds_soilgrids2017 = xr.open_zarr(f'{dir01}ds_prep_soilgrids2017_r025.zarr')

# Soilgrids 2020
ds_soilgrids2020 = xr.open_zarr(f'{dir01}ds_prep_soilgrids2020_r025.zarr')

# Worldclim
ds_worldclim = xr.open_zarr(f'{dir01}ds_prep_worldclim_r025.zarr')

# Geom90m
ds_geom90m = xr.open_zarr(f'{dir01}ds_prep_geom90m_r025.zarr')

In [ ]:
# List with ecosystem types
list_etype = ['trop_ef', 'trop_df', 'temp_ebf', 'temp_dbf', 'temp_enf',
              'temp_dnf', 'glob_shrub', 'glob_grass', 'glob_tundr', 
              'glob_mount', 'glob_crop', 'glob_past']

# List with covariates
vars_soilgrids2017 = ['soilgrids2017_bdricm', 'soilgrids2017_bdrlog',
                      'soilgrids2017_bdticm']

vars_soilgrids2020 = ['soilgrids2020_cec', 'soilgrids2020_cfvo', 
                      'soilgrids2020_clay', 'soilgrids2020_phh2o', 
                      'soilgrids2020_sand', 'soilgrids2020_silt']

vars_worldclim = ['worldclim_bio1', 'worldclim_bio3', 'worldclim_bio4', 
                  'worldclim_bio5', 'worldclim_bio6', 'worldclim_bio12', 
                  'worldclim_bio13', 'worldclim_bio14', 'worldclim_bio15', 
                  'worldclim_elev']

vars_geom90m = ['geom90m_convergence', 'geom90m_cti', 'geom90m_eastness',
               'geom90m_northness', 'geom90m_slope', 'geom90m_spi']

In [ ]:
# Combine relevant datasets in one dataset
ds_all = xr.merge([ds_land.luh2_land_mask, 
                   ds_soilgrids2017[vars_soilgrids2017], 
                   ds_soilgrids2020[vars_soilgrids2020], 
                   ds_worldclim[vars_worldclim], 
                   ds_geom90m[vars_geom90m]])

# Convert dataset to dataframe
df_all = ds_all \
    .to_dataframe(dim_order=['lat', 'lon'])

# Remove all non-land grid cells
df_all = df_all[df_all.luh2_land_mask == 1]

# Reset index and drop land mask column
df_all = df_all \
    .reset_index() \
    .drop('luh2_land_mask', axis=1)

# Export df_all
df_all.to_parquet(dir02 + 'dd_rf_dbase_r025.parquet') 

df_all